# Module 01: Your First Plot

With the environment confirmed and libraries installed, there is nothing left to set up. This module goes straight into making figures, starting from the most minimal working example possible and building toward loading real experimental data and saving publication-ready files.

## Imports and Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## A minimal plot from hand-typed data

The most direct path to a figure is to define two lists, one for x and one for y, and pass them to Matplotlib. This is not how you will work most of the time, but it is useful to see the core mechanism before adding any complexity.

In [ ]:
x = [0, 1, 2, 3, 4, 5]
y = [0, 1, 4, 9, 16, 25]

fig, ax = plt.subplots()
ax.plot(x, y)
plt.show()

That is a plot of y = x². Six points, connected by a line. The defaults are plain, but the structure is already there.

The line `fig, ax = plt.subplots()` is worth pausing on, because you will see two different styles in Matplotlib code online. Some code calls `plt.plot()` directly without any setup. That works for a quick throwaway figure, but it gives you no handle on the figure's components afterward.

The `fig, ax = plt.subplots()` pattern is different. Think of `fig` as the canvas: it represents the entire figure, including its size and any margins. Think of `ax` as the drawing area on that canvas: the region where your data appears, along with its axes, tick marks, and labels. Once you have `ax` as an object, you can control every part of it explicitly. We will use this pattern throughout the course.

## Generating data with NumPy

Typing data by hand works for six points. For anything larger, or for plotting a mathematical function, NumPy is the right tool. The function `np.linspace(start, stop, n)` generates `n` evenly spaced values between `start` and `stop`. From there, you can compute y values from the entire array at once, with no loop required.

To connect directly to what we just plotted, here is y = x² again, this time generated rather than typed.

In [ ]:
x = np.linspace(0, 5, 6)  # 6 evenly spaced points from 0 to 5
y = x**2                  # NumPy applies the squaring to every element at once

In [ ]:
fig, ax = plt.subplots()
ax.plot(x, y, color='steelblue', linewidth=2)
ax.set_title('y = x² (NumPy version)')  # We can easily add a title
plt.show()

The same curve as before! You can make it smoother by using 200 points instead of 6, without having to type a single x and y value manually. The expression `x**2` operates on the entire array at once: NumPy applies the operation element-wise and returns a new array of the same length. This generalises to any mathematical expression, however complex.

That same mechanism makes it easy to plot functions a scientist would recognise. Here is a Gaussian, the shape that appears in spectral peaks, diffusion profiles, and probability distributions:
$$f(x) = ae^{-\frac{(x-b)^2}{2c^2}}$$
<div style="text-align: center;">
    <b>Variables:</b> $a$ = peak height • $b$ = center position • $c$ = standard deviation • $e$ = Euler's number • $x$ = input value
</div>

In [ ]:
x = np.linspace(-4, 4, 300)  # Wider range to capture the full bell shape
y = np.exp(-x**2 / 2)  # Gaussian centred at 0 with height and std of 1

In [ ]:
fig, ax = plt.subplots()

ax.plot(x, y, color='steelblue', linewidth=2)
ax.set_title('Gaussian shape')
ax.set_xlabel('x')  # We can add a label for the x axis
ax.set_ylabel('exp(−x² / 2)')  # Also for the y axis
plt.show()

The `ax.set_xlabel`, `ax.set_ylabel`, and `ax.set_title` calls follow a consistent pattern: they are all methods on the `ax` object, each taking a string. We will use these in every module from here on.

## Creating the sample dataset

The next two sections load data from files. To avoid asking you to download anything, the cell below creates the dataset and writes it to `data/sample_viscosity.csv`. Run this cell once and the file will be there for the rest of the module.

The dataset represents viscosity measurements of a concentrated polymer solution at different temperatures. Viscosity is reported in millipascal-seconds (mPa·s) and temperature in degrees Celsius. Values decrease with temperature in a pattern consistent with an Arrhenius-type dependence, which you will see modelled properly in Module 04.

In [ ]:
os.makedirs('data', exist_ok=True)

viscosity_data = pd.DataFrame({
    'Temperature_C': [10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80],
    'Viscosity_mPas': [142.3, 118.7, 98.4, 81.2, 67.5, 55.9, 46.3, 38.4, 31.8, 26.4, 21.9, 18.2, 15.1, 12.6, 10.5]
})

viscosity_data.to_csv('data/sample_viscosity.csv', index=False)
print('File written to data/sample_viscosity.csv')

## Loading data from a CSV file

With the file in place, `pd.read_csv()` loads it into a DataFrame in a single line. A DataFrame is pandas' core data structure: a table with named columns that you can index, filter, and manipulate in ways that will become familiar quickly.

In [ ]:
df = pd.read_csv('data/sample_viscosity.csv')
df

The column names come directly from the header row in the CSV file. You access a column by name using square brackets: `df['Temperature_C']` returns a pandas Series, which behaves like an array and can be passed directly to Matplotlib.

In [ ]:
fig, ax = plt.subplots()
ax.plot(df['Temperature_C'], df['Viscosity_mPas'], marker='o')  # We add circular markers at each data point
ax.set_xlabel('Temperature (°C)')
ax.set_ylabel('Viscosity (mPa·s)')
ax.set_title('Viscosity vs. Temperature')
plt.show()

The `marker='o'` argument adds a visible point at each data location. This matters for experimental data, where you want to show the actual measurements rather than imply a continuous function. The line between points is just a visual guide.

## Loading data from an Excel file

Excel files are common in lab environments. The cell below writes an Excel version of the same dataset so you can see how the loading process compares. In practice, you would already have the `.xlsx` file and would skip straight to `pd.read_excel()`.

In [ ]:
viscosity_data.to_excel('data/sample_viscosity.xlsx', index=False)
print('Excel file written to data/sample_viscosity.xlsx')

In [ ]:
df_excel = pd.read_excel('data/sample_viscosity.xlsx')
df_excel.head()  # .head() shows the first five rows

The result is identical to what `pd.read_csv()` returned. `pd.read_excel()` accepts a `sheet_name` argument if your file has multiple sheets; by default it loads the first one. If your Excel file has units or notes in a header row above the column names, you can use the `skiprows` argument to skip past them.

For files with multiple sheets, the call would look like this:

```python
df = pd.read_excel('data/my_file.xlsx', sheet_name='Sheet2')
```

## Saving figures

Matplotlib can save figures to several formats. The format is inferred from the file extension you provide. PNG is a raster format: it stores pixels, so the image quality depends on the resolution you set. SVG and PDF are vector formats: they store the drawing instructions mathematically, so they scale to any size without loss. For a figure that will appear in a journal article or a thesis, always save to SVG or PDF.

In [ ]:
fig, ax = plt.subplots()

# Same plotting lines as before
ax.plot(df['Temperature_C'], df['Viscosity_mPas'], marker='o')
ax.set_xlabel('Temperature (°C)')
ax.set_ylabel('Viscosity (mPa·s)')
ax.set_title('Viscosity vs. Temperature')

# Save in three formats
os.makedirs('result', exist_ok=True)
fig.savefig('result/viscosity_plot.png', dpi=150, bbox_inches='tight')
fig.savefig('result/viscosity_plot.svg', bbox_inches='tight')
fig.savefig('result/viscosity_plot.pdf', bbox_inches='tight')

print('Figures saved.')
plt.show()

A few things to note about these arguments. `dpi=150` sets the resolution for the PNG: 150 dots per inch is adequate for screen sharing and slides; 300 is the standard for print. The `dpi` argument has no effect on SVG or PDF output because those formats are resolution-independent. `bbox_inches='tight'` trims whitespace from around the figure, which prevents awkward padding when you insert the image into a document. It is almost always the right choice.

The saved files will appear in your `result/` folder in the Colab Files panel. You can download any of them by right-clicking and selecting Download.